In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from glob import glob
import re
import nltk

In [2]:
nltk_resources = [
    'tokenizers/punkt', 
    'averaged_perceptron_tagger_eng',
    'corpora/stopwords', 
    'help/tagsets'
]

for resource in nltk_resources:
    try:
        nltk.data.find(resource)
    except IndexError:
        nltk.download(resource)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


# LIB and CORPUS tables

In [3]:
LIB = pd.concat([
    pd.read_csv('/kaggle/input/notebooks/isabellouisedelgado/high-gothic-corpus/LIB_high_gothic.csv', index_col='book_id'),
    pd.read_csv('/kaggle/input/notebooks/isabellouisedelgado/classical-gothic-corpus/LIB_classical_gothic.csv', index_col='book_id'),
    pd.read_csv('/kaggle/input/notebooks/isabellouisedelgado/victorian-gothic-corpus/LIB_victorian_gothic.csv', index_col='book_id'),
    pd.read_csv('/kaggle/input/notebooks/isabellouisedelgado/modern-gothic-corpus/LIB_modern_gothic.csv', index_col='book_id'),
]).sort_index()

In [4]:
LIB['period'] = LIB.index.map({
    696:   'early gothic',
    3268:  'early gothic',
    5182:  'early gothic',
    53685: 'classical gothic',
    41445: 'classical gothic',
    6087:  'classical gothic',
    345:   'victorian gothic',
    43:    'victorian gothic',
    1952:  'victorian gothic',
    175: 'modern gothic',
    12122: 'modern gothic',
    20180856: 'modern gothic'
})

In [5]:
LIB

,source_file_path,author,title,chap_pat,clip_pats,period
book_id,,,,,,
43,/kaggle/input/datasets/isabellouisedelgado/vic...,"STEVENSON, ROBERT",DR JEKYLL AND MR HYDE,^(?:STORY OF THE DOOR|SEARCH FOR MR\. HYDE|DR\...,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",victorian gothic
175,/kaggle/input/datasets/isabellouisedelgado/mod...,"LEROUX, GASTON",THE PHANTOM OF THE OPERA,^Chapter\s+[IVXLC]+\s+.*$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",modern gothic
345,/kaggle/input/datasets/isabellouisedelgado/vic...,"STOKER, BRAM",DRACULA,^CHAPTER\s+[IVXLCM]+$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",victorian gothic
696,/kaggle/input/datasets/isabellouisedelgado/hig...,"WALPOLE, HORACE",CASTLE OF OTRANTO,^CHAPTER\s+[IVXLCM]+\.$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",early gothic
1952,/kaggle/input/datasets/isabellouisedelgado/vic...,"GILMAN, CHARLOTTE",THE YELLOW WALLPAPER,^The Yellow Wallpaper$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",victorian gothic
3268,/kaggle/input/datasets/isabellouisedelgado/hig...,"RADCLIFFE, ANN",THE MYSTERIES OF UDOLPHO,^(?:VOLUME\s+\d+|\ CHAPTER\s+[IVXLCM]+)$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",early gothic
5182,/kaggle/input/datasets/isabellouisedelgado/hig...,"REEVE, CLARA",THE OLD ENGLISH BARON,^THE OLD ENGLISH BARON: A GOTHIC STORY\.$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",early gothic
6087,/kaggle/input/datasets/isabellouisedelgado/cla...,"POLIDORI, JOHN",THE VAMPYRE,^\s*THE VAMPYRE\.\s*$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",classical gothic
12122,/kaggle/input/datasets/isabellouisedelgado/mod...,"JACOBS, WW",THE MONKEYS PAW,^\s*[IVXLCM]+\.\s*$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",modern gothic


In [6]:
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

CORPUS = pd.concat([
    pd.read_csv('/kaggle/input/notebooks/isabellouisedelgado/high-gothic-corpus/CORPUS_high_gothic.csv', index_col=OHCO),
    pd.read_csv('/kaggle/input/notebooks/isabellouisedelgado/classical-gothic-corpus/CORPUS_classical_gothic.csv', index_col=OHCO),
    pd.read_csv('/kaggle/input/notebooks/isabellouisedelgado/victorian-gothic-corpus/CORPUS_victorian_gothic.csv', index_col=OHCO),
    pd.read_csv('/kaggle/input/notebooks/isabellouisedelgado/modern-gothic-corpus/CORPUS_modern_gothic.csv', index_col=OHCO),
])

# Add period column from LIB
CORPUS['period'] = CORPUS.index.get_level_values('book_id').map(LIB['period'])

In [7]:
CORPUS

token_str  pos pos_group  \
book_id  chap_num para_num sent_num token_num                            
696      1        1        0        0           Manfred  NNP        NN   
                                    2            Prince  NNP        NN   
                                    3                of   IN        IN   
                                    4           Otranto  NNP        NN   
                                    6               had  VBD        VB   
...                                                 ...  ...       ...   
20180856 9        119      4        34         whatever  WDT        WD   
                                    35           walked  VBD        VB   
                                    36            there   RB        RB   
                                    38           walked  VBD        VB   
                                    39            alone   RB        RB   

                                               term_str         period  
book_id  chap_num para_num sent_num token_num                           
696      1        1        0        0           manfred   early gothic  
                                    2            prince   early gothic  
                                    3                of   early gothic  
                                    4           otranto   early gothic  
                                    6               had   early gothic  
...                                                 ...            ...  
20180856 9        119      4        34         whatever  modern gothic  
                                    35           walked  modern gothic  
                                    36            there  modern gothic  
                                    38           walked  modern gothic  
                                    39            alone  modern gothic  

[869324 rows x 5 columns]

In [8]:
#add book length to LIB
LIB['book_len'] = CORPUS.groupby('book_id').term_str.count()
LIB.sort_values('book_len')

,source_file_path,author,title,chap_pat,clip_pats,period,book_len
book_id,,,,,,,
12122,/kaggle/input/datasets/isabellouisedelgado/mod...,"JACOBS, WW",THE MONKEYS PAW,^\s*[IVXLCM]+\.\s*$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",modern gothic,4023
1952,/kaggle/input/datasets/isabellouisedelgado/vic...,"GILMAN, CHARLOTTE",THE YELLOW WALLPAPER,^The Yellow Wallpaper$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",victorian gothic,6178
6087,/kaggle/input/datasets/isabellouisedelgado/cla...,"POLIDORI, JOHN",THE VAMPYRE,^\s*THE VAMPYRE\.\s*$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",classical gothic,10991
43,/kaggle/input/datasets/isabellouisedelgado/vic...,"STEVENSON, ROBERT",DR JEKYLL AND MR HYDE,^(?:STORY OF THE DOOR|SEARCH FOR MR\. HYDE|DR\...,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",victorian gothic,25756
696,/kaggle/input/datasets/isabellouisedelgado/hig...,"WALPOLE, HORACE",CASTLE OF OTRANTO,^CHAPTER\s+[IVXLCM]+\.$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",early gothic,34877
5182,/kaggle/input/datasets/isabellouisedelgado/hig...,"REEVE, CLARA",THE OLD ENGLISH BARON,^THE OLD ENGLISH BARON: A GOTHIC STORY\.$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",early gothic,54463
53685,/kaggle/input/datasets/isabellouisedelgado/cla...,"MATURIN, CHARLES",MELMOTH THE WANDERER,^(?:CHAPTER\s+[IVXLCM]+\.|Tale of the Spaniard...,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",classical gothic,57022
20180856,/kaggle/input/datasets/isabellouisedelgado/mod...,"JACKSON, SHIRLEY",THE HAUNTING OF HILL HOUSE,^(?:ONE|TWO|THREE|FOUR|FIVE|SIX|SEVEN|EIGHT|NI...,"['_For Leonard Brown_', 'TRANSCRIBER NOTES']",modern gothic,65454
41445,/kaggle/input/datasets/isabellouisedelgado/cla...,"SHELLEY, MARY",FRANKENSTEIN,^(?:LETTER|CHAPTER)\s+[IVXLCM]+\.$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",classical gothic,72069


In [9]:
#add number of chapters
LIB['n_chaps'] = CORPUS.reset_index()[['book_id','chap_num']]\
    .drop_duplicates()\
    .groupby('book_id').chap_num.count()
LIB

,source_file_path,author,title,chap_pat,clip_pats,period,book_len,n_chaps
book_id,,,,,,,,
43,/kaggle/input/datasets/isabellouisedelgado/vic...,"STEVENSON, ROBERT",DR JEKYLL AND MR HYDE,^(?:STORY OF THE DOOR|SEARCH FOR MR\. HYDE|DR\...,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",victorian gothic,25756,9
175,/kaggle/input/datasets/isabellouisedelgado/mod...,"LEROUX, GASTON",THE PHANTOM OF THE OPERA,^Chapter\s+[IVXLC]+\s+.*$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",modern gothic,83686,26
345,/kaggle/input/datasets/isabellouisedelgado/vic...,"STOKER, BRAM",DRACULA,^CHAPTER\s+[IVXLCM]+$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",victorian gothic,162434,27
696,/kaggle/input/datasets/isabellouisedelgado/hig...,"WALPOLE, HORACE",CASTLE OF OTRANTO,^CHAPTER\s+[IVXLCM]+\.$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",early gothic,34877,5
1952,/kaggle/input/datasets/isabellouisedelgado/vic...,"GILMAN, CHARLOTTE",THE YELLOW WALLPAPER,^The Yellow Wallpaper$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",victorian gothic,6178,1
3268,/kaggle/input/datasets/isabellouisedelgado/hig...,"RADCLIFFE, ANN",THE MYSTERIES OF UDOLPHO,^(?:VOLUME\s+\d+|\ CHAPTER\s+[IVXLCM]+)$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",early gothic,292371,4
5182,/kaggle/input/datasets/isabellouisedelgado/hig...,"REEVE, CLARA",THE OLD ENGLISH BARON,^THE OLD ENGLISH BARON: A GOTHIC STORY\.$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",early gothic,54463,1
6087,/kaggle/input/datasets/isabellouisedelgado/cla...,"POLIDORI, JOHN",THE VAMPYRE,^\s*THE VAMPYRE\.\s*$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",classical gothic,10991,2
12122,/kaggle/input/datasets/isabellouisedelgado/mod...,"JACOBS, WW",THE MONKEYS PAW,^\s*[IVXLCM]+\.\s*$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",modern gothic,4023,3


In [10]:
LIB.to_csv('GothicNovels_LIB.csv')
CORPUS.to_csv('GothicNovels_CORPUS.csv')

# VOCAB table

In [11]:
VOCAB = CORPUS.term_str.value_counts().to_frame('n').sort_index()
VOCAB.index.name = 'term_str'
VOCAB['n_chars'] = VOCAB.index.str.len()
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()
VOCAB['i'] = -np.log2(VOCAB.p)
VOCAB

,n,n_chars,p,i
term_str,,,,
1,31,1,0.000036,14.775338
10,8,2,0.000009,16.729534
100,1,3,0.000001,19.729534
1018,1,4,0.000001,19.729534
1030,2,4,0.000002,18.729534
...,...,...,...,...
æt,1,2,0.000001,19.729534
ætat,1,4,0.000001,19.729534
æther,6,5,0.000007,17.144572


In [12]:
VOCAB['max_pos'] = CORPUS[['term_str','pos']].value_counts().unstack(fill_value=0).idxmax(1)
VOCAB['max_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().unstack(fill_value=0).idxmax(1)
VOCAB

,n,n_chars,p,i,max_pos,max_pos_group
term_str,,,,,,
1,31,1,0.000036,14.775338,CD,CD
10,8,2,0.000009,16.729534,CD,CD
100,1,3,0.000001,19.729534,CD,CD
1018,1,4,0.000001,19.729534,CD,CD
1030,2,4,0.000002,18.729534,CD,CD
...,...,...,...,...,...,...
æt,1,2,0.000001,19.729534,NNP,NN
ætat,1,4,0.000001,19.729534,VBZ,VB
æther,6,5,0.000007,17.144572,NN,NN


In [13]:
sw = pd.DataFrame(nltk.corpus.stopwords.words('english'), columns=['term_str'])
sw = sw.reset_index().set_index('term_str')
sw.columns = ['dummy']
sw.dummy = 1
sw.sample(5)

,dummy
term_str,
have,1
yourselves,1
am,1
i'd,1
they'll,1


In [14]:
VOCAB['stop'] = sw
VOCAB['stop'] = VOCAB['stop'].fillna(0).astype('int')

In [15]:
print("Stopwords:", int(VOCAB.stop.sum()))

Stopwords: 153


In [16]:
from nltk.stem.porter import PorterStemmer
stemmer1 = PorterStemmer()
VOCAB['stem_porter'] = VOCAB.apply(lambda x: stemmer1.stem(x.name), 1)

In [17]:
VOCAB.sample(10) #will add dfidf later when constructing BOW and TDIDF

,n,n_chars,p,i,max_pos,max_pos_group,stop,stem_porter
term_str,,,,,,,,
believed,140,8,0.000161,12.600251,VBD,VB,0,believ
facehappily,1,11,0.000001,19.729534,NN,NN,0,facehappili
forbids,10,7,0.000012,16.407606,VBZ,VB,0,forbid
doorway,35,7,0.000040,14.600251,NN,NN,0,doorway
fellowcreature,1,14,0.000001,19.729534,NN,NN,0,fellowcreatur
files,2,5,0.000002,18.729534,NNS,NN,0,file
make,620,4,0.000713,10.453410,VB,VB,0,make
illlit,3,6,0.000003,18.144572,JJ,JJ,0,illlit
contraction,1,11,0.000001,19.729534,NN,NN,0,contract


In [18]:
VOCAB.to_csv('GothicNovels_VOCAB.csv') 